In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_7.py ──
"""
Shared infrastructure for MLFP03 Exercise 7 — Kailash Workflows, DataFlow
Persistence, Hyperparameter Search, and Model Registry.

Contains: leak-free dataset loading, fixed dev/test frames, FeatureSchema
construction, registry setup and artefact loading, metric computation, the
production quality gate, illustrative ROI helpers, DB URL resolution and
pipeline-audit utilities. Technique-specific code (workflow node wiring, search space
definitions, registry lifecycle transitions) lives in the per-technique files.

Available after ``uv sync`` from any directory.
"""

import os
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml.types import FeatureField, FeatureSchema


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()

RANDOM_SEED: int = 42
TARGET_COLUMN: str = "default"
DATASET_NAME: str = "sg_credit_scoring"
DATASET_FILE: str = "sg_credit_scoring.parquet"

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer (Exercise 4 screens for it).
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")

# Output directory for artefacts (audit trails, evaluation tables)
OUTPUT_DIR = Path("outputs") / "mlfp03_ex7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DataFlow persistence URL. SQLite is hermetic — every student gets a fresh
# DB file per run. In production we would read this from the environment.
#
# NOTE: The modern dataflow sqlite adapter interprets `sqlite:///relative`
# as an absolute `/relative` path (breaking old sqlite URL conventions).
# We therefore resolve to an absolute path and use the `sqlite:////abs`
# four-slash form so the behaviour is identical on every working dir.
_DB_ABS_PATH = (OUTPUT_DIR / "mlfp03_models.db").resolve()
DB_URL: str = os.environ.get(
    "MLFP03_EX7_DB_URL", f"sqlite:///{_DB_ABS_PATH.as_posix()}"
)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════


def load_credit_frame() -> pl.DataFrame:
    """Load the Singapore credit scoring dataset as a polars DataFrame.

    Columns: demographic + bureau features, with ``default`` (0/1) target.
    ``CREDIT_NON_FEATURE_COLUMNS`` (row ID + post-outcome leak) are dropped
    here so every downstream split, schema and pipeline inherits the fix.
    """
    loader = MLFPDataLoader()
    return loader.load("mlfp02", DATASET_FILE).drop(CREDIT_NON_FEATURE_COLUMNS)


def prepare_credit_frames(
    credit: pl.DataFrame | None = None, *, seed: int = RANDOM_SEED
) -> tuple[pl.DataFrame, pl.DataFrame, list[str]]:
    """Return ``(dev_frame, test_frame, feature_columns)`` for honest model selection.

    ``dev_frame`` (80%) is everything model SELECTION may touch —
    hyperparameter search, grid baselines, early stopping; TrainingPipeline
    carves its own validation holdout out of it. ``test_frame`` (20%) is
    touched exactly once, to report the chosen model. The 20% is held out
    BEFORE the preprocessing is fitted, so its rows shape no imputation or
    encoding statistic. Both carry a unique
    ``application_id`` so they satisfy the same ``FeatureSchema``.
    """
    if credit is None:
        credit = load_credit_frame()

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target=TARGET_COLUMN,
        test_size=0.2,
        seed=seed,
        normalize=False,
        categorical_encoding="ordinal",
    )
    n_dev = result.train_data.height
    dev = result.train_data.with_columns(
        pl.int_range(0, n_dev, dtype=pl.Int64).alias("application_id")
    )
    test = result.test_data.with_columns(
        pl.int_range(n_dev, n_dev + result.test_data.height, dtype=pl.Int64).alias(
            "application_id"
        )
    )
    feature_columns = [
        c for c in dev.columns if c not in (TARGET_COLUMN, "application_id")
    ]
    return dev, test, feature_columns


def credit_feature_schema(feature_columns: list[str]) -> FeatureSchema:
    """Build a FeatureSchema matching ``prepare_credit_frames`` output."""
    return FeatureSchema(
        name="credit_model_input",
        features=[FeatureField(name=f, dtype="float64") for f in feature_columns],
        entity_id_column="application_id",
    )


async def build_training_registry(db_url: str | None = None):
    """Create + initialise a kailash-ml ModelRegistry for TrainingPipeline.

    Returns ``(registry, connection_manager)``. Caller owns ``connection_manager.close()``.
    """
    from kailash.db import ConnectionManager
    from kailash_ml import ModelRegistry

    conn = ConnectionManager(db_url or DB_URL)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return registry, conn


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def compute_classification_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray
) -> dict[str, float]:
    """Return accuracy, f1, auc_roc, auc_pr, log_loss."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
    }


def print_metric_block(title: str, metrics: dict[str, float]) -> None:
    print(f"\n=== {title} ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


# ════════════════════════════════════════════════════════════════════════
# QUALITY GATE — what a candidate model must clear before production
# ════════════════════════════════════════════════════════════════════════
# Both thresholds are relative to what "no skill" looks like on THIS data:
#   - AUC-ROC 0.5 is random ranking; we demand a clear margin above it.
#   - AUC-PR (average precision) of a random scorer equals the default
#     rate (~13% here); we demand at least twice that.
# Leak-free LightGBM on this dataset lands around AUC 0.77 / AUC-PR 0.37,
# so a sound model passes and a broken one (e.g. shuffled labels) fails.

PROMOTION_MIN_AUC: float = 0.75
PROMOTION_MIN_AP_MULTIPLE: float = 2.0  # × default rate


def promotion_gate(metrics: dict[str, float], default_rate: float) -> dict[str, Any]:
    """Evaluate the production quality gate.

    ``metrics`` uses ``compute_classification_metrics`` names
    (``auc_roc``, ``auc_pr``) computed on rows the model never trained on.
    Returns a JSON-serialisable dict: thresholds, observed values, verdict.
    """
    min_ap = PROMOTION_MIN_AP_MULTIPLE * default_rate
    auc = float(metrics["auc_roc"])
    ap = float(metrics["auc_pr"])
    return {
        "auc_roc": auc,
        "auc_pr": ap,
        "min_auc_roc": PROMOTION_MIN_AUC,
        "min_auc_pr": min_ap,
        "promote": bool(auc >= PROMOTION_MIN_AUC and ap >= min_ap),
    }


async def load_registered_model(registry: Any, name: str, version: int) -> Any:
    """Load a model artefact that THIS course's pipeline registered.

    The registry stores the fitted estimator as pickle bytes. Unpickling
    executes code, so only ever load artefacts you trained yourself.
    """
    import pickle

    artifact = await registry.load_artifact(name, version)
    return pickle.loads(artifact)


# Metrics TrainingPipeline can compute from its holdout predictions. In
# kailash-ml 2.2.2 the engine's evaluator does not pass predicted
# probabilities to the metric registry, so "average_precision",
# "log_loss" and "brier_score_loss" are skipped there; the exercises
# compute those with ``compute_classification_metrics`` on the
# registered model's probabilities instead.
ENGINE_METRICS: list[str] = ["accuracy", "f1", "auc"]


# ════════════════════════════════════════════════════════════════════════
# APPLY-PHASE BUSINESS ASSUMPTIONS (R9B) — ILLUSTRATIVE
# ════════════════════════════════════════════════════════════════════════
# Round numbers for a HYPOTHETICAL Singapore retail bank's unsecured
# lending book. They are teaching assumptions, not figures from any real
# lender, regulator or report — replace them with your institution's own
# numbers. Only the model-quality inputs (defaults caught, metrics) are
# measured by the exercise code itself.

ILLUSTRATIVE_BANK: dict[str, Any] = {
    "applications_per_year": 200_000,
    "avg_exposure_sgd": 18_000.0,  # average principal per approved loan
    "lgd": 0.65,  # loss given default on unsecured retail
    "review_budget": 0.10,  # share of applications the credit team can review
    "evidence_hours_per_retrain": 160.0,  # manual evidence-pack assembly
    "analyst_hourly_sgd": 120.0,
    "retrains_per_year": 12,
}


def defaults_caught_at_budget(
    y_true: np.ndarray, y_proba: np.ndarray, budget: float | None = None
) -> int:
    """Defaults among the top ``budget`` share of applications by model score.

    This is how a credit team actually uses a score: it can only review a
    fixed share of applications, so a better model is one that puts more
    true defaults into that reviewed slice.
    """
    budget = ILLUSTRATIVE_BANK["review_budget"] if budget is None else budget
    k = max(1, int(round(budget * len(y_true))))
    top = np.argsort(-np.asarray(y_proba))[:k]
    return int(np.asarray(y_true)[top].sum())


def annual_loss_avoided(extra_defaults_caught: float, n_scored: int) -> float:
    """Scale extra defaults caught on ``n_scored`` test rows to a year (S$)."""
    b = ILLUSTRATIVE_BANK
    per_year = extra_defaults_caught * b["applications_per_year"] / n_scored
    return float(per_year * b["avg_exposure_sgd"] * b["lgd"])


def evidence_prep_savings() -> float:
    """Annual analyst cost of hand-assembling model evidence packs (S$)."""
    b = ILLUSTRATIVE_BANK
    return float(
        b["evidence_hours_per_retrain"] * b["analyst_hourly_sgd"] * b["retrains_per_year"]
    )


def headline_roi_text(loss_avoided_sgd: float | None = None) -> str:
    """Plain-text ROI block for the Apply phases (ILLUSTRATIVE assumptions).

    ``loss_avoided_sgd`` must come from a measured comparison (see
    03_hyperparameter_search.py); when it is not supplied the line says so
    instead of inventing a number.
    """
    b = ILLUSTRATIVE_BANK
    audit = evidence_prep_savings()
    lines = [
        "  (illustrative assumptions for a hypothetical Singapore retail bank)",
        f"  Applications scored:  {b['applications_per_year']:,} / yr",
        f"  Retrains:             {b['retrains_per_year']} / yr",
        f"  Evidence-pack prep:   ~S${audit/1e3:,.0f}k / yr of analyst time that a",
        "                        persisted, replayable pipeline removes",
    ]
    if loss_avoided_sgd is None:
        lines.append(
            "  Loss avoided:         not claimed here — measured in "
            "03_hyperparameter_search.py"
        )
    else:
        lines.append(
            f"  Loss avoided:         ~S${loss_avoided_sgd/1e6:,.2f}M / yr "
            "(measured lift × assumptions)"
        )
    return "\n".join(lines)


# ════════════════════════════════════════════════════════════════════════
# PIPELINE AUDIT HELPERS
# ════════════════════════════════════════════════════════════════════════


def audit_trail_row(
    *,
    stage: str,
    detail: str,
    run_id: str,
) -> dict[str, Any]:
    """Structured audit row used by the orchestrated pipeline."""
    return {"stage": stage, "detail": detail, "run_id": run_id}




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 7.2: DataFlow Persistence with @db.model
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Declare a database table with `@db.model` (schema = Python class)
#   - Let DataFlow auto-migrate the schema — no hand-written DDL
#   - Run the full CRUD cycle with `db.express`: create, read
#     (find_one / read / list / count), update and delete
#   - Persist a TrainingPipeline evaluation next to its registry reference
#   - Understand why governed ML needs a first-class result store
#
# PREREQUISITES: 01_workflow_builder.py
# ESTIMATED TIME: ~35 min
#
# 5-PHASE R10:
#   1. Theory     — why "write metrics to a database" matters
#   2. Build      — @db.model classes for evaluations + artefacts
#   3. Train      — train once, then create / update / delete rows
#   4. Visualise  — read the persisted records back
#   5. Apply      — a bank's model evidence store
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import json
from typing import Any

from dataflow import DataFlow
from kailash_ml import TrainingPipeline
from kailash_ml.engines.training_pipeline import EvalSpec, ModelSpec
from kailash_ml.interop import to_sklearn_input



## THEORY — Why persist metrics?


In [ ]:
# The `print()` statement is the cheapest, worst form of observability
# for ML. Months after a loan was declined, a model validator, internal
# auditor or supervisor can ask:
#
#   "Show me the model that was used for this decision, the version,
#    the exact metrics on the evaluation set at the time of promotion,
#    and the hyperparameters the training run used."
#
# If your answer is "let me grep some log files", you have already lost.
# The defensible answer is a database table — queryable, indexed by
# model name + version — that records every evaluation, every artefact
# and every review decision. DataFlow turns that table from a week of
# SQL-migration work into a decorated Python class, and `db.express`
# gives you the whole CRUD cycle as one-line async calls.



## TASK 2 — BUILD @db.model tables


``@db.model`` auto-generates the primary key ``id`` from the plain
    ``id: int`` annotation (plus created_at / updated_at timestamps).
    Plain class annotations ARE the schema; defaults are ordinary
    Python defaults.


Registry reference + human review status for each model version.


In [ ]:
# TODO: create the DataFlow instance for this exercise's database
# Hint: DataFlow(<database url from the shared helper>)
db = ____


@db.model
class CreditEvaluation:

    id: int
    model_name: str
    model_version: int
    dataset: str
    accuracy: float
    f1_score: float
    auc_roc: float
    auc_pr: float
    log_loss_val: float
    train_rows: int
    feature_count: int
    hyperparameters: str = "{}"


@db.model
class CreditModelArtifact:

    # TODO: declare the primary key and the columns below as plain annotations.
    # Hint: `name: type` — @db.model turns annotations into the table schema;
    # review_status defaults to "pending"
    id: ____
    model_name: str
    version: int
    registry_ref: str
    review_status: str = ____
    created_by: str = "mlfp03_ex7"


MODEL_NAME = "credit_default_dataflow"
HYPERPARAMETERS: dict[str, Any] = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 5,
    "num_leaves": 31,
    "min_child_samples": 40,
    "random_state": RANDOM_SEED,
    "verbose": -1,
}



## TASK 3 — TRAIN once, then CREATE / UPDATE / DELETE rows


Returns (test metrics, registry version, training rows, n_features).


CREATE the evaluation + artefact rows, UPDATE the review, DELETE a scratch row.


READ: by id, by filter, the full list, and a count.


In [ ]:
# Training goes through TrainingPipeline (fit + holdout evaluation +
# registry entry in one engine call) on the dev frame; the registered
# model is then scored on the test frame. Every DataFlow call below is
# awaited inside ONE event loop — the DataFlow connection pool belongs to
# the loop that opened it, so one asyncio.run() drives the whole file.


async def train_with_engine() -> tuple[dict[str, float], int, int, int]:
    dev, test, feature_cols = prepare_credit_frames()
    registry, conn = await build_training_registry()
    try:
        pipeline = TrainingPipeline(feature_store=None, registry=registry)
        result = await pipeline.train(
            data=dev,
            schema=credit_feature_schema(feature_cols),
            model_spec=ModelSpec(
                model_class="lightgbm.LGBMClassifier",
                framework="lightgbm",
                hyperparameters=HYPERPARAMETERS,
            ),
            eval_spec=EvalSpec(
                metrics=ENGINE_METRICS, split_strategy="holdout", test_size=0.2
            ),
            experiment_name=MODEL_NAME,
        )
        if result.model_version is None:
            raise RuntimeError("TrainingPipeline did not register the model")
        version = int(result.model_version.version)
        model = await load_registered_model(registry, MODEL_NAME, version)
    finally:
        await conn.close()

    X_test, y_test, _ = to_sklearn_input(
        test, feature_columns=feature_cols, target_column=TARGET_COLUMN
    )
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = compute_classification_metrics(y_test, (y_proba >= 0.5).astype(int), y_proba)
    return metrics, version, dev.height, len(feature_cols)


async def persist_review_cleanup(
    metrics: dict[str, float], version: int, rows: int, n_features: int
) -> dict[str, Any]:
    await db.initialize()

    # CREATE — returns the written fields plus rows_affected (no id echo)
    # TODO: insert one CreditEvaluation row
    # Hint: db.express.create("<Model name>", {<column>: <value>, ...})
    created_eval = await db.express.____(
        "CreditEvaluation",
        {
            "model_name": MODEL_NAME,
            "model_version": version,
            "dataset": DATASET_NAME,
            "accuracy": metrics["accuracy"],
            "f1_score": metrics["f1"],
            "auc_roc": metrics["auc_roc"],
            "auc_pr": metrics["auc_pr"],
            "log_loss_val": metrics["log_loss"],
            "train_rows": rows,
            "feature_count": n_features,
            "hyperparameters": json.dumps(HYPERPARAMETERS),
        },
    )
    created_artifact = await db.express.create(
        "CreditModelArtifact",
        {
            "model_name": MODEL_NAME,
            "version": version,
            "registry_ref": f"model_registry://{MODEL_NAME}/v{version}",
        },
    )

    # READ the row we just wrote (find_one filters on any columns)
    # TODO: read back the artefact row by filtering on model_name and version
    # Hint: db.express.find_one("<Model>", {<column>: <value>, ...})
    artifact = ____

    # UPDATE — a reviewer signs off this version
    # TODO: set review_status to "approved" on that row (by its id)
    # Hint: db.express.update("<Model>", <id>, {<column>: <new value>})
    reviewed = ____

    # DELETE — a smoke-test row written by a CI check must not linger
    await db.express.create(
        "CreditEvaluation",
        {
            "model_name": "ci_smoke_test",
            "model_version": 0,
            "dataset": "synthetic",
            "accuracy": 0.0,
            "f1_score": 0.0,
            "auc_roc": 0.5,
            "auc_pr": 0.0,
            "log_loss_val": 0.0,
            "train_rows": 0,
            "feature_count": 0,
        },
    )
    smoke = await db.express.find_one("CreditEvaluation", {"model_name": "ci_smoke_test"})
    # TODO: delete the smoke-test row by its id
    # Hint: db.express.delete("<Model>", <id>) returns True on success
    deleted = ____
    smoke_left = await db.express.count("CreditEvaluation", {"model_name": "ci_smoke_test"})

    return {
        "created_eval": created_eval,
        "created_artifact": created_artifact,
        "artifact_id": artifact["id"],
        "reviewed": reviewed,
        "deleted": deleted,
        "smoke_left": smoke_left,
    }


async def read_back(version: int) -> dict[str, Any]:
    evaluation = await db.express.find_one(
        "CreditEvaluation", {"model_name": MODEL_NAME, "model_version": version}
    )
    artifact = await db.express.find_one(
        "CreditModelArtifact", {"model_name": MODEL_NAME, "version": version}
    )
    # TODO: read the same artefact row back by primary key
    # Hint: db.express.read("<Model>", <id>)
    by_id = ____
    return {
        "evaluation": evaluation,
        "artifact": by_id,
        "all_evaluations": await db.express.list(
            "CreditEvaluation", filter={"model_name": MODEL_NAME}
        ),
        "n_artifacts": await db.express.count(
            "CreditModelArtifact", {"model_name": MODEL_NAME}
        ),
    }


async def run_all() -> dict[str, Any]:
    metrics, version, rows, n_features = await train_with_engine()
    try:
        crud = await persist_review_cleanup(metrics, version, rows, n_features)
        stored = await read_back(version)
    finally:
        await db.close_async()
    return {"metrics": metrics, "version": version, **crud, **stored}


out  = await run_all()
print_metric_block("Test-frame metrics of the registered model — ready to persist", out["metrics"])



### Checkpoint


In [ ]:
# db.express.create returns {<fields...>, "rows_affected": 1}; the
# auto-generated id is visible when the row is read back.
assert out["created_eval"].get("rows_affected") == 1, "Task 3: one evaluation row"
assert out["created_artifact"].get("rows_affected") == 1, "Task 3: one artefact row"
assert out["reviewed"]["review_status"] == "approved", "Task 3: update must persist"
assert out["deleted"] is True and out["smoke_left"] == 0, "Task 3: delete must remove the row"
print("\n[ok] Checkpoint passed — create / update / delete verified\n")



## TASK 4 — VISUALISE persisted records by reading them back


In [ ]:
# Any persistence code you cannot READ BACK is a black hole. The audit
# test is "I wrote it, I can find it, I can filter it, I can count it."

ev, art = out["evaluation"], out["artifact"]
print(f"=== {MODEL_NAME} v{out['version']} as stored ===")
print(
    f"  CreditEvaluation id={ev['id']}: AUC-ROC={ev['auc_roc']:.4f} "
    f"AUC-PR={ev['auc_pr']:.4f} train_rows={ev['train_rows']:,} features={ev['feature_count']}"
)
print(
    f"  CreditModelArtifact id={art['id']}: {art['registry_ref']} "
    f"review_status={art['review_status']}"
)
print(f"\n=== Evaluation history for {MODEL_NAME} ({len(out['all_evaluations'])} runs) ===")
for row in out["all_evaluations"]:
    print(
        f"  v{row['model_version']:<3} AUC-PR={row['auc_pr']:.4f}  "
        f"written {row.get('created_at', '?')}"
    )
print(f"  artefact rows for {MODEL_NAME}: {out['n_artifacts']}")



### Checkpoint


In [ ]:
assert abs(ev["auc_pr"] - out["metrics"]["auc_pr"]) < 1e-9, (
    "Task 4: the stored AUC-PR must equal the engine's value"
)
assert art["review_status"] == "approved", "Task 4: read-back must see the update"
print("\n[ok] Checkpoint passed — persisted rows read back intact\n")



## TASK 5 — APPLY: A Bank's Model Evidence Store


In [ ]:
# SCENARIO (illustrative): before this file, a Singapore bank's model
# validation team rebuilt an "evidence pack" by hand for every retrain —
# copying metrics out of notebooks, chasing which hyperparameters were
# used, emailing for sign-off. With the two tables above, the evidence
# pack is a query: evaluation metrics + registry reference + review
# status for any model version, written at the moment it happened.
print("\n" + "=" * 70)
print("  APPLY: Model Evidence Store")
print("=" * 70)
print(headline_roi_text())
print(
    "\n  The evidence-pack line above is what persistence buys: the rows"
    "\n  you just wrote replace the manual reconstruction."
)



## REFLECTION


[x] Declared database tables with the @db.model decorator
  [x] Ran the full db.express CRUD cycle: create, find_one/read/list/count,
      update, delete
  [x] Stored a TrainingPipeline evaluation next to its registry reference
  [x] Recorded a human review decision as an UPDATE, not an email

  KEY INSIGHT: If your metrics only exist in a print() call, they don't
  exist from an auditor's perspective. DataFlow turns "write it to a DB"
  from a week of schema work into a decorated class and a few awaits.

  Next: 03_hyperparameter_search.py — use Bayesian search to improve the
  metrics you just learned how to persist.

  DB URL: {DB_URL}


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

